<a href="https://colab.research.google.com/github/stauntonjr/local_llm_notebooks/blob/master/TransMLA_Convert_an_LLM's_GQA_to_MLA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

More details in this article: [TransMLA: Improve Qwen2.5 and Llama 3x LLMs with DeepSeek's Multi-Head Latent Attention](https://kaitchup.substack.com/p/transmla-improve-qwen25-and-llama)

This notebook shows how to convert GQA to MLA using the TransMLA method. The example uses Qwen2.5 but it can also work with Llama 3.x models.

The notebook also performs LoRA fine-tuning, with full fine-tuning for the self-attention, to fully exploit MLA. It can also run fine-tuning with the original GQA for comparison.


# Install

In [ ]:
!pip install --upgrade transformers bitsandbytes peft accelerate datasets trl flash_attn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 103.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.1/342.1 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.1 MB/s eta 0:00:

# Prepare for Using a Custom Modeling Class

In [ ]:
!git clone https://github.com/fxmeng/TransMLA.git

Cloning into 'TransMLA'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 111 (delta 58), reused 82 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 155.35 KiB | 10.36 MiB/s, done.
Resolving deltas: 100% (58/58), done.


Navigate to the model directory since we will need to import from there.

In [ ]:
%cd TransMLA/models/

/content/TransMLA/models


# TransMLA without Absorption

## The Method

In [ ]:
from qwen2.modeling_qwen2 import Qwen2MLAForCausalLM
from qwen2.configuration_qwen2 import Qwen2Config
from transformers import AutoTokenizer, set_seed
import torch
from copy import deepcopy
from tqdm import tqdm

set_seed(1234)

def transMLA(model):
  hidden_size = model.config.hidden_size
  n_heads = model.config.num_attention_heads
  kv_heads = model.config.num_key_value_heads
  head_dim = model.config.hidden_size//model.config.num_attention_heads
  latent_dim = kv_heads * head_dim
  kv_groups = model.config.num_attention_heads // model.config.num_key_value_heads
  model.config.partial_rotary_factor
  # Insert identity matrices
  for name,module in model.named_modules():
    if 'k_up_proj' in name or "v_up_proj" in name:
        weight = torch.stack([torch.eye(latent_dim).reshape(kv_heads, head_dim, latent_dim)]*kv_groups,dim=1).reshape(hidden_size, latent_dim).contiguous().to(module.weight.data.device,module.weight.data.dtype)
        if 'k_up_proj' in name:
            weight = weight.view(hidden_size, kv_heads, head_dim).transpose(1,2).contiguous().view(hidden_size, latent_dim)
        module.weight.data=weight
    elif 'k_proj' in name:
        module.weight.data = module.weight.data.view(kv_heads, head_dim, hidden_size).transpose(0,1).contiguous().view(latent_dim, hidden_size)
        module.bias.data = module.bias.data.view(kv_heads, head_dim).transpose(0,1).contiguous().view(latent_dim)
  for name,module in model.named_modules():
    if name.endswith("self_attn"):
        # Orthogonal q_proj and k_up_proj
        k_up_weight = deepcopy(module.k_up_proj.weight.data).reshape(n_heads, head_dim, latent_dim) # (n_heads, head_dim, latent_dim)
        q_weight = deepcopy(module.q_proj.weight.data).reshape(n_heads, head_dim, hidden_size) # (n_heads, head_dim, hidden_size)
        if module.q_proj.bias is not None:
            q_weight = torch.cat([q_weight,deepcopy(module.q_proj.bias.data).reshape(n_heads, head_dim, 1)],dim=-1)
        q_k_up = torch.einsum("hdc,hdD->hcD",k_up_weight, q_weight) # (n_heads, latent_dim, hidden_size), rank<=head_dim
        U,S,V = torch.svd_lowrank(q_k_up.float(), head_dim, niter=16) # U(n_heads, latent_dim, head_dim), S(n_heads, head_dim), V(n_heads, hidden_size, head_dim)
        U = U.to(q_k_up.dtype)
        S = S.to(q_k_up.dtype)
        V = V.to(q_k_up.dtype)
        US_sqrt = torch.einsum('hLd,hd->hdL',U,torch.sqrt(S)) # (n_heads, head_dim, latent_dim)
        S_sqrtV = torch.einsum('hd,hDd->hdD',torch.sqrt(S),V) # (n_heads, head_dim, hidden_size)
        if module.q_proj.bias is not None:
            module.q_proj.bias.data = S_sqrtV[:,:,-1].reshape(-1).contiguous()
            S_sqrtV = S_sqrtV[:,:,:-1]
        module.k_up_proj.weight.data = US_sqrt.reshape(n_heads*head_dim, latent_dim).contiguous()
        module.q_proj.weight.data = S_sqrtV.reshape(n_heads*head_dim, hidden_size).contiguous()

        # Orthogonal o_proj and v_up_proj
        v_up_weight = deepcopy(module.v_up_proj.weight.data).reshape(n_heads, head_dim, latent_dim)
        o_weight = deepcopy(module.o_proj.weight.data).reshape(hidden_size, n_heads, head_dim)
        v_up_o = torch.einsum("hdc,Dhd->hcD",v_up_weight, o_weight) # (n_heads, latent_dim, hidden_size), rank<=head_dim
        U,S,V = torch.svd_lowrank(v_up_o.float(), head_dim, niter=16) # U(n_heads, latent_dim, head_dim), S(n_heads, head_dim), V(n_heads, hidden_size, head_dim)
        U = U.to(v_up_o.dtype)
        S = S.to(v_up_o.dtype)
        V = V.to(v_up_o.dtype)
        US_sqrt = torch.einsum('hLd,hd->hdL',U,torch.sqrt(S)) # (n_heads, head_dim, latent_dim)
        S_sqrtV = torch.einsum('hd,hDd->Dhd',torch.sqrt(S),V) # (hidden_size, n_heads, head_dim)
        module.v_up_proj.weight.data = US_sqrt.reshape(hidden_size, latent_dim).contiguous()
        module.o_proj.weight.data = S_sqrtV.reshape(hidden_size, n_heads*head_dim).contiguous()
  return model

## Fine-Tuning

In [ ]:
import torch, os, multiprocessing
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed
)
from trl import SFTTrainer, SFTConfig
set_seed(1234)

compute_dtype = torch.bfloat16
attn_implementation = 'sdpa'

def fine_tune(model_name, batch_size=4, gradient_accumulation_steps=8, absorb=False):

  tokenizer = AutoTokenizer.from_pretrained(model_name)
  tokenizer.pad_token = "<|image_pad|>"
  tokenizer.pad_token_id = 151655
  tokenizer.padding_side = 'right'

  ds = load_dataset("HuggingFaceTB/smoltalk", "all", split="train[:50000]")



  model = Qwen2MLAForCausalLM.from_pretrained(
            model_name, device_map={"": 0}, torch_dtype=torch.bfloat16, attn_implementation=attn_implementation, partial_rotary_factor=2, rope_repeat=True
  )
  print(model)

  model = transMLA(model)

  print(model)

  model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':True})
  output_dir = "lora_mla_smoltalk"

  if absorb:
    modules_to_save = ['k_proj', 'v_proj', 'o_proj', 'q_proj']
  else:
    modules_to_save = ['k_proj', 'k_up_proj', 'v_up_proj','v_proj', 'o_proj', 'q_proj']


  peft_config = LoraConfig(
          lora_alpha=16,
          lora_dropout=0.05,
          r=16,
          bias="none",
          task_type="CAUSAL_LM",
          target_modules= ["gate_proj", "down_proj", "up_proj"],
          modules_to_save= modules_to_save
  )




  training_arguments = SFTConfig(
          output_dir=output_dir,
          #eval_strategy="steps",
          #do_eval=True,
          optim="paged_adamw_8bit",
          per_device_train_batch_size=batch_size,
          gradient_accumulation_steps=gradient_accumulation_steps,
          #per_device_eval_batch_size=batch_size,
          log_level="debug",
          #save_strategy="epoch",
          logging_steps=25,
          learning_rate=2e-5,
          bf16 = True,
          eval_steps=25,
          num_train_epochs=1,
          lr_scheduler_type="linear",
          max_seq_length=512,
          report_to="none"
  )

  trainer = SFTTrainer(
          model=model,
          train_dataset=ds,
          #eval_dataset=ds['test'],
          peft_config=peft_config,
          processing_class=tokenizer,
          args=training_arguments,
  )

  #--code by Unsloth: https://colab.research.google.com/drive/1Ys44kVvmeZtnICzWz0xgpRnrIOjZAuxp?usp=sharing#scrollTo=pCqnaKmlO1U9

  gpu_stats = torch.cuda.get_device_properties(0)
  start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
  max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
  print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
  print(f"{start_gpu_memory} GB of memory reserved.")

  trainer_ = trainer.train()


  used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
  used_memory_for_trainer= round(used_memory - start_gpu_memory, 3)
  used_percentage = round(used_memory         /max_memory*100, 3)
  trainer_percentage = round(used_memory_for_trainer/max_memory*100, 3)
  print(f"{trainer_.metrics['train_runtime']} seconds used for training.")
  print(f"{round(trainer_.metrics['train_runtime']/60, 2)} minutes used for training.")
  print(f"Peak reserved memory = {used_memory} GB.")
  print(f"Peak reserved memory for training = {used_memory_for_trainer} GB.")
  print(f"Peak reserved memory % of max memory = {used_percentage} %.")
  print(f"Peak reserved memory for training % of max memory = {trainer_percentage} %.")
  print("-----")
  #----


In [ ]:
fine_tune("Qwen/Qwen2.5-1.5B-Instruct", batch_size=1, gradient_accumulation_steps=16)

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.72k [00:00<?, ?B/s]

...

Step,Training Loss
25,1.181200
50,1.012500
75,1.026900
100,1.023900
125,1.015900
150,0.984200
175,0.995400
200,0.994300
225,0.956800
250,0.973400


Saving model checkpoint to lora_mla_smoltalk/checkpoint-500
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
Model config Qwen2Config {
  "absorb": false,
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "kv_dropout": 0.0,
  "latent_dim_factor": 1,
  "max_position_embeddings": 32768,
  "max_window_layers": 21,
  "model_type": "qwen2",
  "num_attention_heads": 12,
  "num_hidden_layers": 28,
  "num_key_value_heads": 2,
  "partial_rotary_factor": 1,
  "rms_norm_eps": 1e-06,
  "rope_repeat": false,
  "rope_scaling": null,
  "rope_theta": 1000000.0,
  "sliding_window": null,
  "tie_word_embeddings": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.49.0",
  "use_c

Step,Training Loss
25,1.181200
50,1.012500
75,1.026900
100,1.023900
125,1.015900
150,0.984200
175,0.995400
200,0.994300
225,0.956800
250,0.973400


Saving model checkpoint to lora_mla_smoltalk/checkpoint-3000
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
Model config Qwen2Config {
  "absorb": false,
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "kv_dropout": 0.0,
  "latent_dim_factor": 1,
  "max_position_embeddings": 32768,
  "max_window_layers": 21,
  "model_type": "qwen2",
  "num_attention_heads": 12,
  "num_hidden_layers": 28,
  "num_key_value_heads": 2,
  "partial_rotary_factor": 1,
  "rms_norm_eps": 1e-06,
  "rope_repeat": false,
  "rope_scaling": null,
  "rope_theta": 1000000.0,
  "sliding_window": null,
  "tie_word_embeddings": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.49.0",
  "use_

15280.871 seconds used for training.
254.68 minutes used for training.
Peak reserved memory = 5.287 GB.
Peak reserved memory for training = 1.715 GB.
Peak reserved memory % of max memory = 23.857 %.
Peak reserved memory for training % of max memory = 7.739 %.
-----


# TransMLA with Absorption

## The Method

In [ ]:
from qwen2.modeling_qwen2 import Qwen2MLAForCausalLM
from qwen2.configuration_qwen2 import Qwen2Config
from transformers import AutoTokenizer, set_seed
import torch
from copy import deepcopy
from tqdm import tqdm

set_seed(1234)

def transMLA(model):
  hidden_size = model.config.hidden_size
  n_heads = model.config.num_attention_heads
  kv_heads = model.config.num_key_value_heads
  head_dim = model.config.hidden_size//model.config.num_attention_heads
  latent_dim = kv_heads * head_dim
  kv_groups = model.config.num_attention_heads // model.config.num_key_value_heads
  model.config.partial_rotary_factor
  # Insert identity matrices
  for name,module in model.named_modules():
    if 'k_up_proj' in name or "v_up_proj" in name:
        weight = torch.stack([torch.eye(latent_dim).reshape(kv_heads, head_dim, latent_dim)]*kv_groups,dim=1).reshape(hidden_size, latent_dim).contiguous().to(module.weight.data.device,module.weight.data.dtype)
        if 'k_up_proj' in name:
            weight = weight.view(hidden_size, kv_heads, head_dim).transpose(1,2).contiguous().view(hidden_size, latent_dim)
        module.weight.data=weight
    elif 'k_proj' in name:
        module.weight.data = module.weight.data.view(kv_heads, head_dim, hidden_size).transpose(0,1).contiguous().view(latent_dim, hidden_size)
        module.bias.data = module.bias.data.view(kv_heads, head_dim).transpose(0,1).contiguous().view(latent_dim)
  for name,module in model.named_modules():
    if name.endswith("self_attn"):
        # Orthogonal q_proj and k_up_proj
        k_up_weight = deepcopy(module.k_up_proj.weight.data).reshape(n_heads, head_dim, latent_dim) # (n_heads, head_dim, latent_dim)
        q_weight = deepcopy(module.q_proj.weight.data).reshape(n_heads, head_dim, hidden_size) # (n_heads, head_dim, hidden_size)
        if module.q_proj.bias is not None:
            q_weight = torch.cat([q_weight,deepcopy(module.q_proj.bias.data).reshape(n_heads, head_dim, 1)],dim=-1)
        q_k_up = torch.einsum("hdc,hdD->hcD",k_up_weight, q_weight) # (n_heads, latent_dim, hidden_size), rank<=head_dim
        U,S,V = torch.svd_lowrank(q_k_up.float(), head_dim, niter=16) # U(n_heads, latent_dim, head_dim), S(n_heads, head_dim), V(n_heads, hidden_size, head_dim)
        U = U.to(q_k_up.dtype)
        S = S.to(q_k_up.dtype)
        V = V.to(q_k_up.dtype)
        US_sqrt = torch.einsum('hLd,hd->hdL',U,torch.sqrt(S)) # (n_heads, head_dim, latent_dim)
        S_sqrtV = torch.einsum('hd,hDd->hdD',torch.sqrt(S),V) # (n_heads, head_dim, hidden_size)
        if module.q_proj.bias is not None:
            module.q_proj.bias.data = S_sqrtV[:,:,-1].reshape(-1).contiguous()
            S_sqrtV = S_sqrtV[:,:,:-1]
        module.k_up_proj.weight.data = US_sqrt.reshape(n_heads*head_dim, latent_dim).contiguous()
        module.q_proj.weight.data = S_sqrtV.reshape(n_heads*head_dim, hidden_size).contiguous()

        # Orthogonal o_proj and v_up_proj
        v_up_weight = deepcopy(module.v_up_proj.weight.data).reshape(n_heads, head_dim, latent_dim)
        o_weight = deepcopy(module.o_proj.weight.data).reshape(hidden_size, n_heads, head_dim)
        v_up_o = torch.einsum("hdc,Dhd->hcD",v_up_weight, o_weight) # (n_heads, latent_dim, hidden_size), rank<=head_dim
        U,S,V = torch.svd_lowrank(v_up_o.float(), head_dim, niter=16) # U(n_heads, latent_dim, head_dim), S(n_heads, head_dim), V(n_heads, hidden_size, head_dim)
        U = U.to(v_up_o.dtype)
        S = S.to(v_up_o.dtype)
        V = V.to(v_up_o.dtype)
        US_sqrt = torch.einsum('hLd,hd->hdL',U,torch.sqrt(S)) # (n_heads, head_dim, latent_dim)
        S_sqrtV = torch.einsum('hd,hDd->Dhd',torch.sqrt(S),V) # (hidden_size, n_heads, head_dim)
        module.v_up_proj.weight.data = US_sqrt.reshape(hidden_size, latent_dim).contiguous()
        module.o_proj.weight.data = S_sqrtV.reshape(hidden_size, n_heads*head_dim).contiguous()


  for name,module in model.named_modules():
      if name.endswith("self_attn"):
          # Absorb k_up_proj into q_proj
          k_up_weight = deepcopy(module.k_up_proj.weight.data).reshape(n_heads, head_dim, latent_dim) # (n_heads, head_dim, latent_dim)
          q_weight = deepcopy(module.q_proj.weight.data).reshape(n_heads, head_dim, hidden_size) # (n_heads, head_dim, hidden_size)
          if module.q_proj.bias is not None:
              q_weight = torch.cat([q_weight,deepcopy(module.q_proj.bias.data).reshape(n_heads, head_dim, 1)],dim=-1)
          q_k_up = torch.einsum("hdc,hdD->hcD",k_up_weight, q_weight) # (n_heads, latent_dim, hidden_size), rank<=head_dim
          q_proj = torch.nn.Linear(hidden_size, n_heads*latent_dim, bias=(module.q_proj.bias is not None))
          q_proj = q_proj.to(device=module.q_proj.weight.device, dtype=module.q_proj.weight.dtype)
          if module.q_proj.bias is not None:
              q_proj.bias.data = q_k_up[:,:,-1].reshape(-1).contiguous()
              q_k_up = q_k_up[:,:,:-1]
          q_proj.weight.data = q_k_up.reshape(n_heads*latent_dim, hidden_size).contiguous()
          setattr(module, "q_proj", q_proj)
          delattr(module, "k_up_proj")
          # Absorb v_up_proj into o_proj
          v_up_weight = deepcopy(module.v_up_proj.weight.data).reshape(n_heads, head_dim, latent_dim) # (n_heads, head_dim, latent_dim)
          o_weight = deepcopy(module.o_proj.weight.data).reshape(hidden_size, n_heads, head_dim) # (n_heads, head_dim, hidden_size)
          v_up_o = torch.einsum("hdc,Dhd->Dhc",v_up_weight, o_weight) # (hidden_size, n_heads, latent_dim), rank<=head_dim
          o_proj = torch.nn.Linear(n_heads*latent_dim, hidden_size, bias=(module.o_proj.bias is not None))
          o_proj = o_proj.to(device=module.o_proj.weight.device, dtype=module.o_proj.weight.dtype)
          o_proj.weight.data = v_up_o.reshape(hidden_size, n_heads*latent_dim).contiguous()
          if module.o_proj.bias is not None:
              o_proj.bias.data = module.o_proj.bias
          setattr(module, "o_proj", o_proj)
          delattr(module, "v_up_proj")
          module.absorb = True
  return model

## Fine-Tuning

In [ ]:
fine_tune("Qwen/Qwen2.5-1.5B-Instruct", batch_size=1, gradient_accumulation_steps=16, absorb=True)

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.72k [00:00<?, ?B/s]

...

Using auto half precision backend
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Currently training with a batch size of: 1
The following columns in the training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: source, text. If source, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 50,000
  Num Epochs = 1
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 16
  Total optimization steps = 3,125
  Number of trainable parameters = 300,468,224


GPU = NVIDIA L4. Max memory = 22.161 GB.
7.443 GB of memory reserved.


Step,Training Loss
25,1.118300
50,0.942800
75,0.994500
100,0.999900
125,0.992400
150,0.963800
175,0.974500
200,0.973000
225,0.935600
250,0.954600


Saving model checkpoint to lora_mla_smoltalk/checkpoint-500
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
Model config Qwen2Config {
  "absorb": false,
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "kv_dropout": 0.0,
  "latent_dim_factor": 1,
  "max_position_embeddings": 32768,
  "max_window_layers": 21,
  "model_type": "qwen2",
  "num_attention_heads": 12,
  "num_hidden_layers": 28,
  "num_key_value_heads": 2,
  "partial_rotary_factor": 1,
  "rms_norm_eps": 1e-06,
  "rope_repeat": false,
  "rope_scaling": null,
  "rope_theta": 1000000.0,
  "sliding_window": null,
  "tie_word_embeddings": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.49.0",
  "use_c

14912.3902 seconds used for training.
248.54 minutes used for training.
Peak reserved memory = 9.154 GB.
Peak reserved memory for training = 1.711 GB.
Peak reserved memory % of max memory = 41.307 %.
Peak reserved memory for training % of max memory = 7.721 %.
-----


# Fine-Tuning with the Original GQA

In [ ]:
import torch, os, multiprocessing
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed
)
from trl import SFTTrainer, SFTConfig
set_seed(1234)

compute_dtype = torch.bfloat16
attn_implementation = 'sdpa'

def fine_tune(model_name, batch_size=4, gradient_accumulation_steps=8):

  tokenizer = AutoTokenizer.from_pretrained(model_name)
  tokenizer.pad_token = "<|image_pad|>"
  tokenizer.pad_token_id = 151655
  tokenizer.padding_side = 'right'

  ds = load_dataset("HuggingFaceTB/smoltalk", "all", split="train[:50000]")




  model = AutoModelForCausalLM.from_pretrained(
            model_name, device_map={"": 0}, torch_dtype=torch.bfloat16, attn_implementation=attn_implementation
  )
  print(model)



  model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':True})
  output_dir = "lora_gqa"

  peft_config = LoraConfig(
          lora_alpha=16,
          lora_dropout=0.05,
          r=16,
          bias="none",
          task_type="CAUSAL_LM",
          target_modules= [ "gate_proj", "down_proj", "up_proj"],
          modules_to_save= ['k_proj',  'v_proj','q_proj', 'o_proj']
  )




  training_arguments = SFTConfig(
          output_dir=output_dir,
          #eval_strategy="steps",
          #do_eval=True,
          optim="paged_adamw_8bit",
          per_device_train_batch_size=batch_size,
          gradient_accumulation_steps=gradient_accumulation_steps,
          #per_device_eval_batch_size=batch_size,
          log_level="debug",
          #save_strategy="epoch",
          logging_steps=25,
          learning_rate=1e-4,
          bf16 = True,
          eval_steps=25,
          num_train_epochs=1,
          lr_scheduler_type="linear",
          max_seq_length=512,
          report_to="none"
  )

  trainer = SFTTrainer(
          model=model,
          train_dataset=ds,
          #eval_dataset=ds['test'],
          peft_config=peft_config,
          processing_class=tokenizer,
          args=training_arguments,
  )

  #--code by Unsloth: https://colab.research.google.com/drive/1Ys44kVvmeZtnICzWz0xgpRnrIOjZAuxp?usp=sharing#scrollTo=pCqnaKmlO1U9

  gpu_stats = torch.cuda.get_device_properties(0)
  start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
  max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
  print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
  print(f"{start_gpu_memory} GB of memory reserved.")

  trainer_ = trainer.train()


  used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
  used_memory_for_trainer= round(used_memory - start_gpu_memory, 3)
  used_percentage = round(used_memory         /max_memory*100, 3)
  trainer_percentage = round(used_memory_for_trainer/max_memory*100, 3)
  print(f"{trainer_.metrics['train_runtime']} seconds used for training.")
  print(f"{round(trainer_.metrics['train_runtime']/60, 2)} minutes used for training.")
  print(f"Peak reserved memory = {used_memory} GB.")
  print(f"Peak reserved memory for training = {used_memory_for_trainer} GB.")
  print(f"Peak reserved memory % of max memory = {used_percentage} %.")
  print(f"Peak reserved memory for training % of max memory = {trainer_percentage} %.")
  print("-----")
  #----
